# Entrenamiento estilo Colab en VS Code

Este notebook ejecuta el pipeline completo (dependencias, entrenamiento, gráficas y TensorBoard) usando los archivos locales del repositorio.

### Flujo recomendado
1. Instala/actualiza dependencias (incluye TensorBoard para visualizar).
2. Configura el dispositivo y el nombre del experimento.
3. Lanza el entrenamiento con TrainingConfig.
4. Visualiza métricas con Matplotlib y abre TensorBoard sin salir de VS Code.

In [2]:
# 1) Dependencias (se ejecuta una sola vez por entorno)
%pip install -q -r requirements.txt tensorboard pandas
%pip install ultralytics
%pip install -r requirements.txt
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ------------ --------------------------- 3.4/11.0 MB 24.8 MB/s eta 0:00:01
   ---------------------------------------  10.7/11.0 MB 33.8 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 29.3 MB/s eta 0:00:00
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# 2) Configuración general y detección automática del dispositivo
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR
while PROJECT_ROOT.parent != PROJECT_ROOT and not (PROJECT_ROOT / "utils").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "utils").exists():
    raise RuntimeError(
        "No se encontró la carpeta 'utils'. Abre el notebook desde el repositorio FractureDetector o ajusta PROJECT_ROOT manualmente."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Usando raíz del proyecto: {PROJECT_ROOT}")

from utils.helpers import timestamp_slug

def resolve_device_preference(device_arg: str = "auto") -> str:
    requested = (device_arg or "auto").strip().lower()
    if requested != "auto":
        return requested

    try:
        import torch

        if torch.cuda.is_available() and torch.cuda.device_count() > 0:
            return "0"
        if hasattr(torch, "backends") and hasattr(torch.backends, "mps"):
            if torch.backends.mps.is_available():
                return "mps"
    except Exception:
        pass

    return "cpu"

EXPERIMENT_NAME = timestamp_slug("notebook")
RUN_DEVICE = resolve_device_preference("auto")
print(f"Ejecutando experimento '{EXPERIMENT_NAME}' en {RUN_DEVICE}")

Usando raíz del proyecto: C:\Users\USUARIO\Desktop\Detección de Fracturas Óseas en Radiografías\FractureDetector
Ejecutando experimento 'notebook-20251117-011712' en cpu


In [9]:
# 3) Entrenamiento con la configuración deseada
from src.training import TrainingConfig, train_detector

DATA_YAML = PROJECT_ROOT / "data/raw/data.yaml"
PROJECT_RUNS = PROJECT_ROOT / "runs/notebooks"
MODELS_DIR = PROJECT_ROOT / "models"
EXPERIMENT_NAME = "notebook-20241116-xxxx"  # run de 50 épocas

config = TrainingConfig(
    data_yaml=DATA_YAML,
    model_variant="yolov8n.pt",
    epochs=200,
    batch=4,
    imgsz=512,
    device=RUN_DEVICE,
    project_dir=PROJECT_RUNS,
    experiment_name=EXPERIMENT_NAME,
    export_path=MODELS_DIR / f"{EXPERIMENT_NAME}.pt",
)

export_path = train_detector(config)
run_dir = Path(config.project_dir) / config.experiment_name
print(f"Pesos exportados: {export_path}")
print(f"Resultados y métricas en: {run_dir}")

Ultralytics 8.3.228  Python-3.13.3 torch-2.9.1+cpu CPU (AMD Ryzen 5 4600H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\USUARIO\Desktop\Deteccin de Fracturas seas en Radiografas\FractureDetector\data\raw\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=None, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=notebook-20241116-xxxx, nbs=64, nms=False, opset=Non

KeyboardInterrupt: 

In [3]:
# 4) Gráficas rápidas con Matplotlib
import pandas as pd
import matplotlib.pyplot as plt

results_path = run_dir / "results.csv"
if not results_path.exists():
    raise FileNotFoundError(f"No se encontró {results_path}")

results_df = pd.read_csv(results_path)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results_df.plot(
    x="epoch",
    y=["train/box_loss", "train/cls_loss", "train/dfl_loss"],
    ax=axes[0],
    title="Pérdidas de entrenamiento",
)
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")

results_df.plot(
    x="epoch",
    y=["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"],
    ax=axes[1],
    title="Métricas de validación",
)
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Valor")
plt.tight_layout()
plt.show()

NameError: name 'run_dir' is not defined

In [ ]:
# 5) TensorBoard integrado (ejecuta tras tener logs en runs/notebooks)
%load_ext tensorboard
%tensorboard --logdir runs/notebooks --host 0.0.0.0 --port 6006